# Testes via ollama

In [1]:
from glm_based_event_analysis.link_prediction.in_context_learning import OllamaLinkPredictor, OpenRouterLinkPredictor, vLLMLinkPredictor, LlamaCppLinkPredictor

In [2]:
#MODEL = "gemma3:4b"
MODEL = "google/gemma-4-E4B-it"

In [3]:
#model = OllamaLinkPredictor(model_name=MODEL, generation_params={'temperature': 0.0, 'seed': 2026})
#model._debug = True

In [4]:
# model = OpenRouterLinkPredictor(generation_params={'temperature': 0.0, 'seed': 2026})
# model._debug = True

In [ ]:
model = vLLMLinkPredictor(model_name=MODEL, tokenizer=MODEL, generation_params={'temperature': 0.0, 'seed': 2026, "max_tokens": 256})
model._debug = True

INFO 06-27 15:31:24 [utils.py:278] non-default args: {'tokenizer': 'google/gemma-4-E4B-it', 'max_model_len': 2048, 'disable_log_stats': True, 'model': 'google/gemma-4-E4B-it'}


INFO 06-27 15:31:25 [model.py:617] Resolved architecture: Gemma4ForConditionalGeneration
INFO 06-27 15:31:25 [model.py:1752] Using max model len 2048
INFO 06-27 15:31:25 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 06-27 15:31:25 [config.py:100] Gemma4 model has heterogeneous head dimensions (head_dim=256, global_head_dim=512). Forcing TRITON_ATTN backend to prevent mixed-backend numerical divergence.
INFO 06-27 15:31:25 [vllm.py:977] Asynchronous scheduling is enabled.
INFO 06-27 15:31:25 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=4148121) INFO 06-27 15:31:51 [core.py:112] Initializing a V1 LLM engine (v0.22.1) with config: model='google/gemma-4-E4B-it', speculative_config=None, tokenizer='google/gemma-4-E4B-it', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=tor

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(EngineCore pid=4148121) INFO 06-27 15:31:54 [default_loader.py:397] Loading weights took 1.59 seconds
(EngineCore pid=4148121) INFO 06-27 15:31:55 [gpu_model_runner.py:5132] Model loading took 15.3 GiB memory and 2.536643 seconds
(EngineCore pid=4148121) INFO 06-27 15:31:55 [gpu_model_runner.py:6136] Encoder cache will be initialized with a budget of 8192 tokens, and profiled with 3 video items of the maximum feature size.
(EngineCore pid=4148121) INFO 06-27 15:32:06 [backends.py:1089] Using cache directory: /home/kenzosaki/.cache/vllm/torch_compile_cache/bf30702809/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=4148121) INFO 06-27 15:32:06 [backends.py:1148] Dynamo bytecode transform time: 1.46 s
(EngineCore pid=4148121) INFO 06-27 15:32:07 [backends.py:292] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 0.922 s
(EngineCore pid=4148121) INFO 06-27 15:32:07 [decorators.py:311] Directly load AOT compilation from path /home/kenzosaki/.ca

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:02<00:00, 20.84it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:01<00:00, 28.92it/s]


(EngineCore pid=4148121) INFO 06-27 15:32:14 [gpu_model_runner.py:6456] Graph capturing finished in 4 secs, took 0.71 GiB
(EngineCore pid=4148121) INFO 06-27 15:32:14 [gpu_worker.py:619] CUDA graph pool memory: 0.71 GiB (actual), 0.79 GiB (estimated), difference: 0.08 GiB (11.4%).
(EngineCore pid=4148121) INFO 06-27 15:32:14 [jit_monitor.py:54] Kernel JIT monitor activated — Triton JIT compilations during inference will be logged as warnings.
(EngineCore pid=4148121) INFO 06-27 15:32:14 [core.py:302] init engine (profile, create kv cache, warmup model) took 18.86 s (compilation: 2.59 s)
(EngineCore pid=4148121) INFO 06-27 15:32:14 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])


(EngineCore pid=4148121) WARNING 06-27 15:32:21 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _compute_slot_mapping_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


In [6]:
#model = LlamaCppLinkPredictor(generation_params={'temperature': 0.0, 'seed': 2026, "max_tokens": 256, "cache_prompts": True})

# Carregando grafo base e conjunto de analise

In [7]:
import networkx as nx
import json
import pickle
import json

In [8]:
def prepare_pairs(edges: list[dict], G: nx.Graph) -> list[tuple[dict, dict, str]]:
    pairs = []
    for edge in edges:
        u = edge['u']
        node_attr = G.nodes[u]
        u_data = {
            "what": node_attr["what"],
            "when": node_attr["when"].strftime("%Y-%m-%d-%H:%M:%S"),
            "where": node_attr["where"],
            "who": node_attr["who"],
            "why": node_attr["why"],
            "how": node_attr["how"],
        }
        v = edge['v']
        node_attr = G.nodes[v]
        v_data = {
            "what": node_attr["what"],
            "when": node_attr["when"].strftime("%Y-%m-%d-%H:%M:%S"),
            "where": node_attr["where"],
            "who": node_attr["who"],
            "why": node_attr["why"],
            "how": node_attr["how"],
        }
        label = edge['label']
        pairs.append((u_data, v_data, label))
    return pairs

In [9]:
G = pickle.load(open("/exp_local/kenzosaki/data/iptc/graphs/politics_event_graph.pkl", "rb"))

In [10]:
eval_edges = json.load(open("/exp_local/kenzosaki/data/iptc/labels/politics_labels.json", "r"))

In [11]:
edges = prepare_pairs(eval_edges, G)
len(edges)

5382

In [12]:
# para teste
edges = edges[:10]

# Testando predição de link

In [13]:
from tqdm.notebook import tqdm

In [14]:
def run_ollamma_link_prediction(pairs: list[tuple[dict, dict, str]], use_fake_neighborhood: bool = False) -> list[dict]:

    responses = []
    for u_data, v_data, label in tqdm(pairs, desc="Querying LLM for link prediction"):
        if use_fake_neighborhood:
            response = model.predict_link(u_data, v_data, [])
        else:
            response = model.predict_link(u_data, v_data)
        if response.success:
            responses.append(response) # para coincidir com a saida

    # retorna a pred e o label
    return responses

In [15]:
def run_vllm_link_prediction(pairs: list[tuple[dict, dict, str]], use_fake_neighborhood: bool = False) -> list[dict]:

    responses = []
    events_u = []
    events_v = []
    neighborhoods_u = []

    for u_data, v_data, label in tqdm(pairs, desc="Querying LLM for link prediction"):
        neighbourhood_u = [] if use_fake_neighborhood else None
        events_u.append(u_data)
        events_v.append(v_data)
        neighborhoods_u.append(neighbourhood_u)
    
    responses = model.predict_links(events_u, events_v, neighborhoods_u)

    # retorna a pred e o label
    return responses

In [16]:
def run_parallel_link_prediction(pairs: list[tuple[dict, dict, str]], use_fake_neighborhood: bool = False) -> list[dict]:

    responses = []
    events_u = []
    events_v = []
    neighborhoods_u = []

    for u_data, v_data, label in tqdm(pairs, desc="Querying LLM for link prediction"):
        neighbourhood_u = [] if use_fake_neighborhood else None
        events_u.append(u_data)
        events_v.append(v_data)
        neighborhoods_u.append(neighbourhood_u)
    
    responses = model.parallel_predict_links(events_u, events_v, neighborhoods_u, n_jobs=3)

    # retorna a pred e o label
    return responses

In [17]:
if isinstance(model, OllamaLinkPredictor):
    preds = run_ollamma_link_prediction(edges, use_fake_neighborhood=True)
elif isinstance(model, vLLMLinkPredictor):
    preds = run_vllm_link_prediction(edges, use_fake_neighborhood=True)
elif isinstance(model, LlamaCppLinkPredictor):
    preds = run_parallel_link_prediction(edges, use_fake_neighborhood=True)

Querying LLM for link prediction:   0%|          | 0/10 [00:00<?, ?it/s]

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


System prompt: You are an event graph link prediction model.
Your task is to analyze a local context extracted from an event graph and decide whether a candidate event should be connected to a given anchor event.
Each event is described using structured 5W1H metadata: what, who, when, where, why, and how.
At the end of the prompt, you will receive:
1. an anchor event A;
2. a candidate event B;
3. a random walk of events preceding the anchor event A, which you may leverage as supporting evidence for determining the relationship between A and B.
You must decide whether there should be a link from A to B in the event graph.
A link means that event B is plausibly related to event A according to semantic, temporal, geographic, or causal continuity.
The random walk events are listed in order, from oldest to newest. Use the random walks to assess whether B fits the established pattern of events when available.
Return only a valid JSON object with the following fields:
{
   "explanation": Stri

Rendering prompts:   0%|          | 0/10 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|██████████| 10/10 [00:02<00:00,  3.80it/s, est. speed input: 2186.14 toks/s, output: 356.63 toks/s]


In [18]:
preds[0]

LinkPredictionOutput(success=True, link=True, explanation='The anchor event A describes Chancellor Olaf Scholz losing a vote of confidence on December 16, 2024. The candidate event B describes Olaf Scholz stepping down from office on December 17, 2024, due to a deepening political crisis. Losing a vote of confidence is a direct precursor and likely cause for a leader to step down, making the events temporally and causally linked.', raw_response='{\n  "explanation": "The anchor event A describes Chancellor Olaf Scholz losing a vote of confidence on December 16, 2024. The candidate event B describes Olaf Scholz stepping down from office on December 17, 2024, due to a deepening political crisis. Losing a vote of confidence is a direct precursor and likely cause for a leader to step down, making the events temporally and causally linked.",\n  "link": true\n}')

# Reportando métricas

In [19]:
from sklearn.metrics import classification_report

In [20]:
edges

[({'what': 'Chancellor Olaf Scholz loses a vote of confidence',
   'when': '2024-12-16-08:00:00',
   'where': 'Germany',
   'who': 'Chancellor Olaf Scholz',
   'why': 'Political instability or loss of parliamentary support',
   'how': 'Through a formal parliamentary vote of confidence'},
  {'what': 'Olaf Scholz steps down from office amidst a deepening political crisis',
   'when': '2024-12-17-08:00:00',
   'where': 'Germany',
   'who': 'Olaf Scholz',
   'why': 'Deepening political crisis in the Western mainstream',
   'how': 'Olaf Scholz steps down from his position'},
  True),
 ({'what': 'Kenya’s Ruto notes Somalia’s political uncertainty as presidential term ends',
   'when': '2026-05-14-07:00:00',
   'where': 'Kenya',
   'who': 'Kenya’s Ruto',
   'why': 'The end of the presidential term in Somalia causing political uncertainty',
   'how': 'Through official observations and statements regarding the political transition'},
  {'what': 'More instability for Somalia as another election 

In [21]:
llm_preds = [pred.link for pred in preds]
labels = [label for (_, _, label) in edges]

In [22]:
# most recent
print("Most recent pairs classification report:")
print(classification_report(labels, llm_preds))

Most recent pairs classification report:
              precision    recall  f1-score   support

        True       1.00      1.00      1.00        10

    accuracy                           1.00        10
   macro avg       1.00      1.00      1.00        10
weighted avg       1.00      1.00      1.00        10

